### Phase 1: Data Modeling & Ingestion (Bronze)

Ingest raw and secure high-sensitivity financial data:

- `transfers_raw`: High-velocity JSON feed of 5,000,000 synthetic fund transfers.
- `account_metadata`: Batch CSV containing account limits and verification status.
- `geo_ip_lookup`: Reference data mapping IP addresses to geographical regions.

#### ->Transfers JSON Data

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

rows = 5000000

transfers = spark.range(rows).withColumn(
    "transfer_id", col("id")
).withColumn(
    "account_id", (rand()*100000).cast("int")
).withColumn(
    "amount", (rand()*20000).cast("double")
).withColumn(
    "ip_address", concat(lit("192.168."),(rand()*255).cast("int"),lit("."),(rand()*255).cast("int"))
).withColumn(
    "transfer_time", current_timestamp()
).drop("id")

In [0]:
transfers.write.format("delta") \
.mode("overwrite") \
.saveAsTable("bronze_transfers_raw")

#### ->Account Meta Data

In [0]:
accounts = spark.range(100000).withColumn(
"account_id",col("id")
).withColumn(
"account_limit",(rand()*50000).cast("double")
).withColumn(
"verified",when(rand()>0.3,"Y").otherwise("N")
).drop("id")

accounts.write.format("delta") \
.mode("overwrite") \
.saveAsTable("bronze_account_metadata")

#### ->Geo IP Lookup

In [0]:
geo = spark.range(1000).withColumn(
"ip_prefix",concat(lit("192.168."),col("id"))
).withColumn(
"country",when(rand()>0.5,"US").otherwise("IN")
).drop("id")

geo.write.format("delta") \
.mode("overwrite") \
.saveAsTable("bronze_geo_ip_lookup")